# DepthMaker backend on a free Colab GPU

Runs the DepthMaker server on Colab's free T4 and exposes it over HTTPS with a
Cloudflare quick tunnel, so the Android app can reach it. **For testing only** —
see the limits at the bottom.

**Before you start:** Runtime → Change runtime type → Hardware accelerator: **T4 GPU**.

Then run the cells in order. Cell 3 takes about 5-10 minutes (weights + model load +
startup self-test). Cell 4 prints the Server URL and token to paste into the app.


## 1. Check you actually got a GPU


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.'


## 2. Install the backend and the model

Uses Colab's preinstalled torch (do not reinstall it — it is multi-GB and already
CUDA-enabled). Downloads the **Small (`vits`)** checkpoint: Apache-2.0, the only one
that is safe for paid client work.


In [ ]:
import os

os.chdir('/content')

!git clone -q -b claude/build-app-apk-cr4a7j https://github.com/foonkoons-cyber/oliv 2>/dev/null || (cd /content/oliv && git fetch -q origin claude/build-app-apk-cr4a7j && git checkout -q claude/build-app-apk-cr4a7j && git pull -q)
!git clone -q https://github.com/DepthAnything/Video-Depth-Anything 2>/dev/null || echo 'model repo already cloned'

# Upstream's requirements.txt pins numpy==1.24.0 and fails to build here; install
# only what the model code actually imports. einops and tqdm are hard imports.
!pip install -q einops easydict imageio imageio-ffmpeg fastapi 'uvicorn[standard]' python-multipart
!pip install -q decord 2>/dev/null || pip install -q eva-decord 2>/dev/null || echo 'decord unavailable - the OpenCV frame reader will be used instead'

# Small model weights (Apache-2.0, commercial-safe).
!mkdir -p /content/Video-Depth-Anything/checkpoints
CKPT = '/content/Video-Depth-Anything/checkpoints/video_depth_anything_vits.pth'
WEIGHTS_URL = 'https://huggingface.co/depth-anything/Video-Depth-Anything-Small/resolve/main/video_depth_anything_vits.pth'
if not (os.path.exists(CKPT) and os.path.getsize(CKPT) > 10 * 1024 * 1024):
    !wget -q --show-progress -O {CKPT} {WEIGHTS_URL}
else:
    print('checkpoint already downloaded')

# Real 1->100 progress (idempotent).
!python /content/oliv/backend/patch_progress.py /content/Video-Depth-Anything

!ls -la /content/Video-Depth-Anything/checkpoints


## 3. Start the server

The first boot loads the model and runs a self-test on a bundled clip, which catches a
model that loaded wrong and would emit a flat grey field. Expect a few minutes.


In [ ]:
import os, secrets, subprocess, time, json, urllib.request

TOKEN = secrets.token_hex(16)
DATA = '/content/depthmaker-data'

env = {**os.environ,
       'VDA_ROOT': '/content/Video-Depth-Anything',
       'DEPTHMAKER_TOKEN': TOKEN,
       'DEPTHMAKER_DATA': DATA,
       'PRELOAD_MODELS': 'vits',
       'MIN_FREE_BYTES': str(2 * 1024**3)}

log = open('/content/server.log', 'w')
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server:app', '--host', '127.0.0.1', '--port', '8000', '--workers', '1'],
    cwd='/content/oliv/backend', env=env, stdout=log, stderr=subprocess.STDOUT)

print('waiting for the model to load and the self-test to pass ...')
deadline = time.time() + 900
ready = False
while time.time() < deadline:
    time.sleep(10)
    if server.poll() is not None:
        print('SERVER EXITED. Log:'); print(open('/content/server.log').read()[-3000:]); break
    try:
        req = urllib.request.Request('http://127.0.0.1:8000/health',
                                     headers={'Authorization': f'Bearer {TOKEN}'})
        with urllib.request.urlopen(req, timeout=10) as r:
            body = json.load(r)
        if body.get('ok'):
            print('server ready:', body); ready = True; break
    except urllib.error.HTTPError as e:
        body = json.load(e)
        print('  worker:', body.get('worker'), body.get('worker_error') or '')
        if body.get('worker') == 'failed':
            print(open('/content/server.log').read()[-3000:]); break
    except Exception:
        print('  still starting ...')

if not ready:
    print('\nServer did not become ready. The tail of /content/server.log above says why.')


## 4. Expose it over HTTPS and get the URL

The app refuses plain `http://`, so the tunnel is required — it also supplies a valid
certificate, which a raw Colab IP could not.


In [ ]:
import re, subprocess, time

import os
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate',
                           '--url', 'http://127.0.0.1:8000'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(2)
    text = open('/content/tunnel.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        url = m.group(0); break

if url:
    print('=' * 62)
    print('Paste these into the app: Settings (gear, top right)')
    print()
    print('  Server URL :', url)
    print('  Token      :', TOKEN)
    print()
    print('Then press Save. Keep this notebook tab open while you use the app.')
    print('=' * 62)
else:
    print('Tunnel did not come up. Log:'); print(open('/content/tunnel.log').read()[-2000:])


## 5. Watch it work (optional)

Run this while a job is processing to see the per-window progress and the validation
gate result.


In [ ]:
!tail -n 40 /content/server.log


## Limits — read these

- **Colab free tier disconnects.** Roughly 90 minutes idle, ~12 hours maximum, and a GPU
  is not guaranteed at busy times. When it drops, rerun cells 3 and 4.
- **The tunnel URL changes every run.** Each `trycloudflare.com` address is temporary, so
  you paste a new Server URL into Settings each session.
- **Anyone with the URL and token can use your GPU.** The token is regenerated per run;
  do not post it anywhere.
- **Testing only.** For real client work, run the backend on a rented GPU box with a
  stable domain (see `backend/README.md`), and set `IDLE_SHUTDOWN_MINUTES=15` so an idle
  instance stops billing you.
- **Speed.** A T4 is far slower than the A100 in the published benchmarks. A 10-second
  clip takes a few minutes; the app's ETA is measured live from the running job, so it
  will tell you the truth rather than quoting a benchmark table.
